In [1]:
import ssl, warnings, requests, time
from bs4 import BeautifulSoup
from io import StringIO
from datetime import datetime

from typing import Union

import pandas as pd
pd.set_option('display.max_rows', None)

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

from src.helpers import get_logger

from requests.adapters import HTTPAdapter
from urllib3.poolmanager import PoolManager
from urllib3.util import Retry

#### Initiliazing the Logger

In [2]:
# Create an instance of CustomLogger with logger name and log directory
logger = get_logger("rrc_scraper",log_to_console=True, level="DEBUG")

# Create an error logger that only logs to file
error_logger = get_logger("rrc_errors", log_to_console=True)

# Start the logger
logger.info(f"################## Logging Started ##################")

[INFO] ################## Logging Started ##################


#### Defining URLs

In [3]:
# Defining URL's
base_url = 'https://webapps2.rrc.texas.gov/EWA/'
query_url = base_url + 'drillingPermitsQueryAction.do'

#### Defining Functions

In [4]:
# Trust Windows root store (includes corporate SSL inspection CA)
try:
    import certifi_win32  # noqa: F401
except Exception:
    pass

DEFAULT_TIMEOUT = 30  # seconds

class TLS12Adapter(HTTPAdapter):
    """Force TLS 1.2; avoid deprecated PROTOCOL_TLSv1_2 unless needed."""
    def __init__(self, *args, **kwargs):
        retries = kwargs.pop("retries", Retry(
            total=3, backoff_factor=0.5,
            status_forcelist=(502, 503, 504),
            allowed_methods=("GET", "POST"),
        ))
        super().__init__(max_retries=retries)

    def init_poolmanager(self, connections, maxsize, block=False, **pool_kwargs):
        # Preferred, non-deprecated path
        try:
            ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
            ctx.minimum_version = ssl.TLSVersion.TLSv1_2
            ctx.maximum_version = ssl.TLSVersion.TLSv1_2
        except Exception:
            # Fallback: silence DeprecationWarning locally
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", category=DeprecationWarning, module="ssl")
                ctx = ssl.SSLContext(getattr(ssl, "PROTOCOL_TLSv1_2"))

        # Relax seclevel if your middlebox enforces legacy ciphers
        try:
            ctx.set_ciphers("DEFAULT:@SECLEVEL=1")
        except Exception:
            pass

        self.poolmanager = PoolManager(
            num_pools=connections, maxsize=maxsize, block=block,
            ssl_context=ctx, **pool_kwargs
        )

def make_office_session() -> requests.Session:
    """
    Creates a Session that:
      - trusts Windows cert store (via certifi_win32 import above)
      - uses system proxies if set (s.trust_env=True)
      - forces TLS 1.2 and adds retries
    """
    s = requests.Session()
    s.trust_env = True  # read HTTPS_PROXY/NO_PROXY if IT sets them
    s.mount("https://webapps2.rrc.texas.gov", TLS12Adapter())
    s.mount("https://", TLS12Adapter())  # fallback for other hosts
    return s


In [5]:
def get_headers() -> dict:

    headers = {
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    'Accept-Encoding': 'gzip, deflate, br',
    'Accept-Language': 'en-US,en;q=0.9',
    'Cache-Control': 'max-age=0',
    'Connection': 'keep-alive',
    'Host': 'webapps2.rrc.texas.gov',
    'Sec-Fetch-Dest': 'document',
    'Sec-Fetch-Mode': 'navigate',
    'Sec-Fetch-Site': 'none',
    'Sec-Fetch-User': '?1',
    'Upgrade-Insecure-Requests': '1',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36',
    'sec-ch-ua': "Google Chrome;v='119', 'Chromium';v='119', 'Not?A_Brand';v='24'",
    'sec-ch-ua-mobile': "?0",
    'sec-ch-ua-platform': "Windows"
    }
    
    return headers


def payload(api:str) -> dict:
         
    """      
    Parameters
    ----------
    api : TYPE, int
        DESCRIPTION. 10 digit API. Strips API of initial '42'.

    Returns
    -------
    Dict.
    """

    # Check the length of API

    try:
        assert len(str(api)) == 14 or len(str(api)) == 10, f"Length of API: {api} don't match with 10 or 14 digit"
        
        if len(str(api)) == 14:
            api = (str(api)[2:10])
        
        elif len(str(api)) == 10:
            api = (str(api)[2:])

    except AssertionError as e:
        error_logger.exception(e)

    payload = {
    'methodToCall': 'search',
    'searchArgs.apiNoHndlr.inputValue': api,
    'searchArgs.operatorNameWildcardHndlr.inputValue': 'beginsWith',
    'searchArgs.leaseNameWildcardHndlr.inputValue': 'beginsWith',
    'searchArgs.fieldNameWildcardHndlr.inputValue': 'beginsWith',
    'searchArgs.surveyNameWildcardHndlr.inputValue': 'beginsWith'
    }

    return payload


def get_data_DrillingPermitQuery(req_url: str, headers: dict, payload: dict) -> requests.Response:
    """
    Returns
    -------
    Request Response (or None if failed).
    """
    try:
        with make_office_session() as s:  # <--- CHANGED
            res_get = s.get(url=req_url, headers=headers, timeout=DEFAULT_TIMEOUT)  # <--- CHANGED
            if res_get.status_code == 200:
                cookies = res_get.cookies
                res_post = s.post(
                    url=query_url, data=payload, cookies=cookies,
                    headers=headers, timeout=DEFAULT_TIMEOUT  # <--- CHANGED
                )
                return res_post  # may be non-200; your caller should check
    except Exception as e:
        error_logger.exception(f"Error occured while requesting {req_url}. Error details:", exc_info=True)


def parse_query_data(post_resp:requests.Response) -> tuple[pd.DataFrame, pd.DataFrame, str, pd.Timestamp, int]:
    """      
    Parameters
    ----------
    post_response : TYPE, Requests Response
        DESCRIPTION. Response from the get_data_DrillingPermitQuery()

    Returns
    -------
    DataFrame.
    """
    soup = BeautifulSoup(post_resp.content,'html.parser')
    web_links = soup.select('a')
    href = [base_url + web_link['href'] for web_link in web_links if 'drillingPermitDetailAction' in web_link['href']]

    df_main = pd.read_html(StringIO(post_resp.text))[9]
    df_main.columns = df_main.iloc[1,:]
    df_main = df_main.drop([0,1]).copy()
    df_main = df_main.reset_index(drop=True).copy()

    df_edit = df_main.copy()
    df_edit['API NO.'] = df_main['API NO.'].str.split().str[0]
    df_edit['HREF'] = href
    df_edit[['Submitted_Dt', 'Approved_Dt']] = df_edit['Status Date'].str.split(expand=True).loc[:,[1,3]]
    df_edit['Submitted_Dt'] = pd.to_datetime(df_edit['Submitted_Dt'])
    df_edit['Approved_Dt'] = pd.to_datetime(df_edit['Approved_Dt'])

    apprvd_Dt_max_idx = df_edit.index[df_edit['Approved_Dt']==df_edit['Approved_Dt'].max()]
    max_apprvd_Dt = df_edit['Approved_Dt'].max()
    href_link = df_edit.loc[apprvd_Dt_max_idx,'HREF'].item()

    return df_main, df_edit, href_link, max_apprvd_Dt, apprvd_Dt_max_idx.item()


def get_data_W1_Form(url_href:str) -> requests.Response:
    """      
    Parameters
    ----------
    url_href : TYPE, Str
        DESCRIPTION. URL from parse_query_data()

    Returns
    -------
    Response.
    """

    with make_office_session() as s:
        res_get = s.get(url=url_href, headers=get_headers(), timeout=DEFAULT_TIMEOUT)
    return res_get


def parse_W1_Form(get_resp:requests.Response) -> pd.DataFrame:
    """      
    Parameters
    ----------
    get_resp : TYPE, Requests Response
        DESCRIPTION. get response from get_data_W1_Form()

    Returns
    -------
    DataFrame
    """
    soup = BeautifulSoup(get_resp.text,'html.parser')
    tables = soup.find_all('table',class_='GroupBox1')

    data_dict = {}

    for count,table in enumerate(tables):
        if (table.find_all('th') is not None):
            for i,(th,td) in enumerate(zip(table.find_all('th'),table.find_all('td'))):
                if th.text == 'Horizontal Wellbore':
                    data_dict[th.text] = td.text
                elif th.text == 'Acres':
                    data_dict[th.text] = td.text

    return pd.DataFrame.from_dict(data_dict, orient='index').T


def formatting_results_df(dataframe:pd.DataFrame) -> pd.DataFrame:
    """
    Formatting dfs_unique_Results DataFrame
    """

    # Formatting the columns
    dataframe.columns = dataframe.columns.str.lower().str.replace(".","").str.replace("#","").str.strip().str.replace(" ","_")

    # Keeping relevant columns
    dataframe = dataframe[['api_no', 'lease', 'well_number', 'filing_purpose', 'amend', 
                           'status', 'submitted_dt','approved_dt', 'horizontal_wellbore', 'acres','href']].copy()
    return dataframe


def results_to_csv(dataframe:pd.DataFrame) -> None:

    try:
        now = datetime.now().strftime('%m_%d_%Y-%H_%M_%S')
        filename: str = f'RRC_Scraper_Results_{now}.csv'
        dataframe.to_csv(filename, index=False)
        
        logger.info(f"Created the {filename} file.")
    except Exception as e:
        error_logger.exception("Error while saving results to CSV file.", exc_info=True)

In [6]:
def main() -> None:

    # Read APIs (from Excel in this version)
    df_api = pd.read_excel(
        r"C:\Users\ApoorvaSaxena\OneDrive - Bandera Group\Desktop\Project AP\02.Python\RRC-Scraper\src\notebooks\TexasAPIs.xlsx",
        dtype={'API_10': str}
    )

    logger.info(f"Running the scraper on {df_api.shape[0]} unique wells.")

    dfs_main = []
    dfs_edit = []
    dfs_unique_Results = []

    headers = get_headers()

    for idx, api in enumerate(df_api['API_10']):
        try:
            logger.info(f"Processing API {api} (Index {idx})")

            payload_api = payload(api=api)

            if idx > 1:
                time.sleep(0.5)

            response_from_DrillingPermit_Query = get_data_DrillingPermitQuery(
                req_url=query_url, headers=headers, payload=payload_api
            )

            if not response_from_DrillingPermit_Query or response_from_DrillingPermit_Query.status_code != 200:
                raise ValueError(f"No valid response for API {api}")

            df_mainPage, df_edit, w1_link, Approved_Dt_max, idx_Max_Aprvd_Dt = parse_query_data(
                post_resp=response_from_DrillingPermit_Query
            )

            if idx > 1:
                time.sleep(0.5)

            response_from_W1_query = get_data_W1_Form(url_href=w1_link)

            df_w1 = parse_W1_Form(get_resp=response_from_W1_query)
            df_w1['Approved_Dt'] = Approved_Dt_max

            df_edit_merge_with_w1 = df_edit.merge(df_w1, on='Approved_Dt', how='left').reset_index(drop=True)

            df_unique_res = df_edit_merge_with_w1[
                df_edit_merge_with_w1['Approved_Dt'] == Approved_Dt_max
            ].reset_index(drop=True).copy()

            dfs_edit.append(df_edit_merge_with_w1)
            dfs_main.append(df_mainPage)
            dfs_unique_Results.append(df_unique_res)

            time.sleep(0.5)

        except Exception as e:
            logger.error(f"Error while processing API {api} at index {idx}. Skipping to next.", exc_info=True)
            continue

    if not dfs_unique_Results:
        logger.warning("No data scraped successfully. Exiting.")
        return

    logger.info(f"Scraping completed for {len(dfs_unique_Results)} out of {df_api.shape[0]} APIs.")

    df_Results = formatting_results_df(dataframe=pd.concat(dfs_unique_Results).reset_index(drop=True))

    logger.info("Saving scraper results to CSV!")
    results_to_csv(dataframe=df_Results)

##### Main

In [7]:
if __name__ == "__main__":

    main()

[INFO] Running the scraper on 15 unique wells.
[INFO] Processing API 4222742533 (Index 0)
[INFO] Processing API 4231746689 (Index 1)
[INFO] Processing API 4231746690 (Index 2)
[INFO] Processing API 4232946306 (Index 3)
[INFO] Processing API 4232946308 (Index 4)
[INFO] Processing API 4232946309 (Index 5)
[INFO] Processing API 4232946310 (Index 6)
[INFO] Processing API 4232946322 (Index 7)
[INFO] Processing API 4232946973 (Index 8)
[INFO] Processing API 4232946975 (Index 9)
[INFO] Processing API 4232947001 (Index 10)
[INFO] Processing API 4232947004 (Index 11)
[INFO] Processing API 4232947264 (Index 12)
[INFO] Processing API 4232947265 (Index 13)
[INFO] Processing API 4232947568 (Index 14)
[INFO] Scraping completed for 15 out of 15 APIs.
[INFO] Saving scraper results to CSV!
[INFO] Created the RRC_Scraper_Results_10_08_2025-23_53_34.csv file.
